# 03 — Execution Simulator (validation notebook)

This is a **validation** notebook, not a place to reimplement the
simulator. All logic is imported from `src.simulator`. It demonstrates,
on one de-identified example route, every mechanical element of the
execution-stage simulation: locked prefix, unserved suffix, p50/p85/p95
travel-time handling, delay20/delay30, lateness, OTD, overtime, RSI, moved
stops, and distance premium -- and confirms that unresolved nodes HARD
FAIL rather than silently falling back to a zero-cost phantom stop.


In [ ]:
import os, sys, json
import pandas as pd
assert 'REPO_ROOT' in dir(), "Run notebook 00 first (or re-execute its setup cells)."
sys.path.insert(0, REPO_ROOT)

from src.data import build_canonical_mapping, load_distance_matrix, load_travel_time_p50_matrix, load_locked_routes, validate_locked_routes_resolve
from src.simulator import SimulationContext, simulate_mixed_route, route_stability, objective_Z
from src.quantiles import P85_MULTIPLIER, P95_MULTIPLIER


## Build the simulation context (public, de-identified inputs)

In [ ]:
DEID_DIR = os.path.join(REPO_ROOT, "data_deidentified")
STOPS_PATH = os.path.join(REPO_ROOT, "data_demo_synthetic", "customer_stops", "synthetic_customer_day_stops.csv")

mapping_df, customers, oversized = build_canonical_mapping(STOPS_PATH)
parent_of = dict(zip(zip(mapping_df['delivery_date'], mapping_df['virtual_stop_id']), mapping_df['parent_physical_node']))

dist_matrix = load_distance_matrix(os.path.join(REPO_ROOT, "data_demo_synthetic", "matrices", "synthetic_distance_matrix.csv"))
p50_matrix = load_travel_time_p50_matrix(os.path.join(REPO_ROOT, "data_demo_synthetic", "matrices", "synthetic_travel_time_p50_matrix.csv"))

ctx = SimulationContext(dist_matrix, p50_matrix, customers, parent_of, P85_MULTIPLIER, P95_MULTIPLIER,
                           sa_config['depot_start_min'], sa_config['operating_window_end_min'])
print("Simulation context built.")


## Load one example locked route (public, de-identified)

In [ ]:
locked_routes_df = pd.read_csv(os.path.join(REPO_ROOT, "data_demo_synthetic", "synthetic_locked_routes.csv"))
example_date = sorted(locked_routes_df['delivery_date'].unique())[0]
example_vehicle = sorted(locked_routes_df[locked_routes_df['delivery_date']==example_date]['vehicle_id'].unique())[0]

route_rows = locked_routes_df[(locked_routes_df['delivery_date']==example_date) &
                               (locked_routes_df['vehicle_id']==example_vehicle)].sort_values('sequence')
full_seq = route_rows['node_id'].tolist()
print(f"Example: date={example_date}, vehicle={example_vehicle}, stops={len(full_seq)}")
print(full_seq)


## Demonstrate: locked prefix vs unserved suffix (execution trigger)

At a trigger fraction of 0.5, the first half of the planned stops are
treated as already executed (frozen); only the remainder is a candidate
for corrective re-sequencing.


In [ ]:
trigger_fraction = 0.5
n_frozen = round(trigger_fraction * len(full_seq))
frozen_flags = [i < n_frozen for i in range(len(full_seq))]
print(f"Trigger fraction={trigger_fraction}: {sum(frozen_flags)} frozen (served), {len(frozen_flags)-sum(frozen_flags)} unserved")
print(f"Frozen (locked) prefix:   {[n for n,f in zip(full_seq, frozen_flags) if f]}")
print(f"Unserved (open) suffix:   {[n for n,f in zip(full_seq, frozen_flags) if not f]}")


## Simulate under each disruption channel: p50 / p85 / p95 / delay20 / delay30

In [ ]:
for disruption in ['p50', 'p85', 'p95', 'delay20', 'delay30']:
    m = simulate_mixed_route(ctx, example_date, full_seq, frozen_flags, disruption)
    print(f"{disruption:8s}: lateness={m['lateness']:7.3f} min, distance={m['distance']:7.3f} km, "
          f"travel_time={m['travel_time']:7.3f} min, overtime={m['overtime']:6.3f} min")


## OTD (on-time delivery %)

In [ ]:
from src.metrics import on_time_counts
num_on_time, total = on_time_counts(ctx, example_date, full_seq, frozen_flags, 'p85')
print(f"OTD under p85: {num_on_time}/{total} = {100*num_on_time/total:.2f}%")


## RSI, moved stops, distance premium (comparing a perturbed continuation to the original)

Demonstrates the stability metrics (Eq. 4a-4c) that the SA method
explicitly optimizes for, using a simple swap on the unserved suffix as
an illustrative "corrective" alternative.


In [ ]:
unserved = [n for n, f in zip(full_seq, frozen_flags) if not f]
served = [n for n, f in zip(full_seq, frozen_flags) if f]

if len(unserved) >= 2:
    perturbed = unserved.copy()
    perturbed[0], perturbed[1] = perturbed[1], perturbed[0]  # illustrative swap
    stab = route_stability(unserved, perturbed)
    moved_stops = sum(1 for a, b in zip(unserved, perturbed) if a != b)
    m_orig = simulate_mixed_route(ctx, example_date, served + unserved, [True]*len(served)+[False]*len(unserved), 'p85')
    m_pert = simulate_mixed_route(ctx, example_date, served + perturbed, [True]*len(served)+[False]*len(perturbed), 'p85')
    dist_premium_pct = 100 * (m_pert['distance'] - m_orig['distance']) / m_orig['distance'] if m_orig['distance'] > 0 else 0.0
    print(f"RSI={stab['RSI']:.4f}, PS={stab['PS']:.4f}, ER={stab['ER']:.4f}")
    print(f"Moved stops: {moved_stops}")
    print(f"Distance premium vs original: {dist_premium_pct:+.3f}%")
else:
    print("Fewer than 2 unserved stops in this example -- stability comparison skipped (not applicable).")


## Hard-fail demonstration: an unresolved node MUST raise, never silently return zero

In [ ]:
try:
    ctx.travel_time(example_date, "DEPOT_1", "NONEXISTENT_NODE_XYZ", 'p50')
    raise RuntimeError("This should never be reached -- hard-fail did not trigger!")
except KeyError as e:
    print(f"Correctly raised KeyError (hard fail, no silent fallback): {e}")


## Expected outputs / integrity checks

In [ ]:
checks = {
    "route_loaded": len(full_seq) > 0,
    "frozen_split_correct": sum(frozen_flags) == n_frozen,
    "p85_lateness_gte_p50": simulate_mixed_route(ctx, example_date, full_seq, frozen_flags, 'p85')['lateness'] >=
                             simulate_mixed_route(ctx, example_date, full_seq, frozen_flags, 'p50')['lateness'] - 1e-6,
    "hard_fail_on_unresolved_node": True,  # verified by the try/except cell above not raising RuntimeError
}
for k, v in checks.items():
    print(f"{'PASS' if v else 'FAIL'}  {k}")
NOTEBOOK_03_STATUS = "PASS" if all(checks.values()) else "FAIL"
print(f"\nNOTEBOOK 03 STATUS: {NOTEBOOK_03_STATUS}")
assert NOTEBOOK_03_STATUS == "PASS"
